# 第 0 章：Transformer 基础

本 Notebook 不下载真实大模型。它用一个很小的 Transformer Block 演示数据如何流动：**Token ID → 向量 → Attention → FFN → 下一个 token 的分数**。

建议依次运行每个单元格。免费 CPU 即可完成。

## 1. 导入 PyTorch 并设置小尺寸

`B` 是一次处理的句子数；`S` 是一句话的 token 数；`D` 是每个 token 向量里的数字个数；`H` 是 Attention 头数。为了便于运行，我们用很小的数字。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
B, S, D, H = 1, 16, 64, 4
D_FF = 128
print(f'B={B}, S={S}, D={D}, H={H}, 每头维度={D // H}')

## 2. 归一化与 Attention

`RMSNorm` 让向量的数值尺度保持稳定。`CausalSelfAttention` 做三件事：

1. 把输入向量一次投影成 Q、K、V；
2. 让每个位置为前面的位置打分；
3. 用因果遮罩挡住未来位置。

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * (x / rms)

class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        assert d % n_head == 0
        self.n_head = n_head
        self.head_dim = d // n_head
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)

    def forward(self, x, show_shapes=False):
        B, S, D = x.shape
        qkv = self.qkv(x).view(B, S, 3, self.n_head, self.head_dim)
        q, k, v = qkv.unbind(2)                 # 每个都是 (B, S, H, d)
        q, k, v = [z.transpose(1, 2) for z in (q, k, v)]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(S, S, device=x.device, dtype=torch.bool))
        weights = F.softmax(scores.masked_fill(~mask, float('-inf')), dim=-1)
        y = (weights @ v).transpose(1, 2).contiguous().view(B, S, D)
        if show_shapes:
            print('Q:', tuple(q.shape), 'K:', tuple(k.shape), 'V:', tuple(v.shape))
            print('attention 分数:', tuple(scores.shape))
        return self.proj(y)

## 3. 把 Attention 和 FFN 组装成一个 Block

真实模型会堆叠几十到上百个 Block。这里仅用一个。`x + ...` 是残差连接：新信息和原信息相加。

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class Block(nn.Module):
    def __init__(self, d, n_head, d_ff):
        super().__init__()
        self.n1 = RMSNorm(d)
        self.attn = CausalSelfAttention(d, n_head)
        self.n2 = RMSNorm(d)
        self.ffn = SwiGLU(d, d_ff)

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.ffn(self.n2(x))

x = torch.randn(B, S, D)
block = Block(D, H, D_FF)
block.attn(x, show_shapes=True)
y = block(x)
print('输入:', tuple(x.shape), '输出:', tuple(y.shape))
assert y.shape == (B, S, D)

## 小练习

1. 把 `H=4` 改成 `H=8`，观察 Q 的形状如何变化。
2. 把 `H=3`，运行并解释为什么会触发 `assert`。
3. 下一章会把这里的 K 和 V 保存起来，避免每次生成时都重算历史 token。